In [1]:
!pip install timm -q

import os, cv2, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.optim as optim, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✓ Device:", device)
print("✓ GPUs:", torch.cuda.device_count())

✓ Device: cuda
✓ GPUs: 2


In [2]:
BASE_DIR   = "/kaggle/input/datasets/organizations/nih-chest-xrays/data/"
CSV_PATH   = "/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv"
SAVE_PATH  = "/kaggle/working/best_hybrid_model.pth"
IMG_SIZE   = 224   # reduced from 320 to save GPU memory
BATCH_SIZE = 16

CLASSES = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration',
    'Mass','Nodule','Pneumonia','Pneumothorax','Consolidation',
    'Edema','Emphysema','Fibrosis','Pleural_Thickening','Hernia'
]

# Build image index
print("Building image index...")
image_dict = {}
for folder in os.listdir(BASE_DIR):
    folder_path = os.path.join(BASE_DIR, folder, 'images')
    if os.path.exists(folder_path):
        for fname in os.listdir(folder_path):
            if fname.endswith('.png'):
                image_dict[fname] = os.path.join(folder_path, fname)

print(f"✓ Total images indexed: {len(image_dict)}")

# Load CSV
df = pd.read_csv(CSV_PATH)
for cls in CLASSES:
    df[cls] = df['Finding Labels'].apply(lambda x: 1 if cls in str(x) else 0)

df['full_path'] = df['Image Index'].map(image_dict)
df = df[df['full_path'].notna()].reset_index(drop=True)
print(f"✓ Total usable images: {len(df)}")

# Split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(f"✓ Train: {len(train_df)} | Val: {len(val_df)}")

# Dataset class
class ChestXrayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df
        self.transform = transform
        self.labels    = df[CLASSES].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = cv2.imread(self.df.loc[idx, 'full_path'], cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img   = clahe.apply(img)
        img   = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img   = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx])

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Weighted sampler
label_matrix   = train_df[CLASSES].values
class_counts   = label_matrix.sum(axis=0).clip(min=1)
sample_weights = torch.tensor(
    label_matrix @ (1.0 / class_counts), dtype=torch.float
)
sampler = WeightedRandomSampler(
    sample_weights, len(sample_weights), replacement=True
)

# Dataloaders
train_loader = DataLoader(
    ChestXrayDataset(train_df, train_transform),
    batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    ChestXrayDataset(val_df, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
print(f"✓ Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print("✓ Data ready!")

Building image index...
✓ Total images indexed: 112120
✓ Total usable images: 112120
✓ Train: 89696 | Val: 22424
✓ Train batches: 5606 | Val batches: 1402
✓ Data ready!


In [3]:
class HybridEfficientNetTransformer(nn.Module):
    def __init__(self, num_classes=14, dropout=0.3):
        super().__init__()

        self.cnn = timm.create_model(
            'efficientnet_b4',
            pretrained=True,
            features_only=True
        )

        with torch.no_grad():
            dummy       = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
            feat_out    = self.cnn(dummy)[-1]
            actual_ch   = feat_out.shape[1]
        print(f"  Detected feature channels: {actual_ch}")

        self.channel_reduce = nn.Sequential(
            nn.Conv2d(actual_ch, 512, kernel_size=1),
            nn.BatchNorm2d(512),
            nn.GELU()
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=512, nhead=8,
            dim_feedforward=1024,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=2
        )

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        features = self.cnn(x)[-1]
        features = self.channel_reduce(features)
        B, C, H, W = features.shape
        tokens   = features.flatten(2).permute(0, 2, 1)
        tokens   = self.transformer(tokens)
        out      = tokens.mean(dim=1)
        out      = self.dropout(out)
        out      = self.classifier(out)
        return out

# Create model
model = HybridEfficientNetTransformer(num_classes=14).to(device)

# Use both GPUs
if torch.cuda.device_count() > 1:
    print(f"✓ Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

total = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✓ Model: Hybrid EfficientNet-B4 + Transformer")
print(f"✓ Parameters: {total:.1f}M")

# Test
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
    out   = model(dummy)
print(f"✓ Output shape: {out.shape}")

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  Detected feature channels: 448
✓ Using 2 GPUs!
✓ Model: Hybrid EfficientNet-B4 + Transformer
✓ Parameters: 21.2M
✓ Output shape: torch.Size([2, 14])


In [4]:
# ── LOAD CHECKPOINT AND RESUME ────────────────────────────────────

checkpoint_path = "/kaggle/input/datasets/heysirii/mini-project-hybrid-model-checkpoint/model/best_hybrid_model.pth"

if os.path.exists(checkpoint_path):
    base_model = model.module if hasattr(model, 'module') else model
    base_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("✓ Checkpoint loaded! Resuming training...")
else:
    print("No checkpoint found — starting fresh")

✓ Checkpoint loaded! Resuming training...


In [ ]:
# Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma

    def forward(self, pred, target):
        bce  = nn.functional.binary_cross_entropy_with_logits(
            pred, target, reduction='none'
        )
        pt   = torch.exp(-bce)
        loss = ((1 - pt) ** self.gamma) * bce
        return loss.mean()

criterion = FocalLoss(gamma=2)

# Evaluate
def evaluate():
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            preds = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.numpy())
    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    aucs = []
    print("\nPer-class AUC:")
    for i, cls in enumerate(CLASSES):
        if all_labels[:, i].sum() > 0:
            auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
            aucs.append(auc)
            print(f"  {cls:<22} {auc:.4f}")
    mean_auc = np.mean(aucs)
    print(f"\n  ► Mean AUC: {mean_auc:.4f}\n")
    return mean_auc

# Optimizer with different LR per layer
for p in model.parameters(): p.requires_grad = True
base_model = model.module if hasattr(model, 'module') else model

optimizer = optim.AdamW([
    {'params': base_model.cnn.parameters(),            'lr': 1e-4},
    {'params': base_model.channel_reduce.parameters(), 'lr': 3e-4},
    {'params': base_model.transformer.parameters(),    'lr': 3e-4},
    {'params': base_model.classifier.parameters(),     'lr': 5e-4},
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr          = [1e-4, 3e-4, 3e-4, 5e-4],
    epochs          = 20,
    steps_per_epoch = len(train_loader),
    pct_start       = 0.3,
    anneal_strategy = 'cos'
)

print("="*55)
print("FULL TRAINING — 2 GPUs — all layers from epoch 1")
print("="*55)

best_auc         = 0
patience_counter = 0
PATIENCE         = 7

for epoch in range(20):
    model.train()
    total_loss = 0
    for i, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if i % 200 == 0:
            print(f"  Ep{epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} avg loss: {avg:.4f}")
    auc = evaluate()

    if auc > best_auc:
        best_auc         = auc
        patience_counter = 0
        torch.save(base_model.state_dict(), SAVE_PATH)
        print(f"  ✓ Best saved! AUC: {best_auc:.4f}")
    else:
        patience_counter += 1
        print(f"  Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("  Early stopping.")
            break

print(f"\n{'='*55}")
print(f"✓ Training complete!")
print(f"✓ Best Mean AUC: {best_auc:.4f}")

FULL TRAINING — 2 GPUs — all layers from epoch 1
  Ep1 | Batch 0/5606 | Loss: 0.0682
  Ep1 | Batch 200/5606 | Loss: 0.0720
  Ep1 | Batch 400/5606 | Loss: 0.0621
  Ep1 | Batch 600/5606 | Loss: 0.0747


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


  Ep1 | Batch 800/5606 | Loss: 0.0773
  Ep1 | Batch 1000/5606 | Loss: 0.0791
  Ep1 | Batch 1200/5606 | Loss: 0.0675
  Ep1 | Batch 1400/5606 | Loss: 0.0667
  Ep1 | Batch 1600/5606 | Loss: 0.0645


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


  Ep1 | Batch 1800/5606 | Loss: 0.0812
  Ep1 | Batch 2000/5606 | Loss: 0.0667
  Ep1 | Batch 2200/5606 | Loss: 0.0649
  Ep1 | Batch 2400/5606 | Loss: 0.0563
  Ep1 | Batch 2600/5606 | Loss: 0.0767
  Ep1 | Batch 2800/5606 | Loss: 0.0705
  Ep1 | Batch 3000/5606 | Loss: 0.0634
  Ep1 | Batch 3200/5606 | Loss: 0.0802
  Ep1 | Batch 3400/5606 | Loss: 0.0605
  Ep1 | Batch 3600/5606 | Loss: 0.0643
  Ep1 | Batch 3800/5606 | Loss: 0.0643
  Ep1 | Batch 4000/5606 | Loss: 0.0743
  Ep1 | Batch 4200/5606 | Loss: 0.0689
  Ep1 | Batch 4400/5606 | Loss: 0.0812


...

  Ep2 | Batch 2600/5606 | Loss: 0.0571
  Ep2 | Batch 2800/5606 | Loss: 0.0554
  Ep2 | Batch 3000/5606 | Loss: 0.0613
  Ep2 | Batch 3200/5606 | Loss: 0.0642


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


  Ep2 | Batch 3400/5606 | Loss: 0.0662
  Ep2 | Batch 3600/5606 | Loss: 0.0880
  Ep2 | Batch 3800/5606 | Loss: 0.0770
  Ep2 | Batch 4000/5606 | Loss: 0.0807
  Ep2 | Batch 4200/5606 | Loss: 0.0741
  Ep2 | Batch 4400/5606 | Loss: 0.0620
  Ep2 | Batch 4600/5606 | Loss: 0.0755
  Ep2 | Batch 4800/5606 | Loss: 0.0730


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


  Ep2 | Batch 5000/5606 | Loss: 0.0710
  Ep2 | Batch 5200/5606 | Loss: 0.0661
  Ep2 | Batch 5400/5606 | Loss: 0.0842
  Ep2 | Batch 5600/5606 | Loss: 0.0628

Epoch 2 avg loss: 0.0689
